# Stage 02: Multimodal VLM (llama3.2-vision) Prompt Engineering & Stochasticity

## Overview
This notebook evaluates the **`llama3.2-vision`** model on biomedical microscopy images in accordance with **Task 1**.
We evaluate an unconstrained naive prompt against an optimized structured descriptive prompt with a strict JSON schema, uncertainty tokens, and temperature stochasticity analysis ($T=0.7$ vs $T=0.0$).

> **Note on Model Execution & Runtime Fallback:**
> The primary specified model is `llama3.2-vision`. If the local Ollama backend encounters a runtime architecture limitation with `llama3.2-vision` (such as the known Windows Ollama `mllama` backend issue), the pipeline automatically and gracefully reroutes inference to `llava:7b` while preserving strict, transparent data provenance (`model`, `requested_model`, `fallback_used`).


In [8]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path
import json

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.vlm import (
    run_naive_vlm,
    run_structured_vlm,
    evaluate_vlm_stochasticity,
    NAIVE_PROMPT,
    STRUCTURED_VLM_PROMPT
)


## 1. Naive Prompting with llama3.2-vision (Baseline)
In this test, llama3.2-vision is queried with an open-ended prompt without domain constraints.


In [9]:
data_dir = PROJECT_ROOT / "data" / "nuclei_dataset"
if not data_dir.exists():
    data_dir = PROJECT_ROOT / "nuclei_dataset"

sample_img = data_dir / "train" / "images" / "train_000.png"
naive_res = run_naive_vlm(sample_img, model="llama3.2-vision", temperature=0.7)

print("VLM Execution Provenance:")
print(f"  Requested Model : {naive_res.get('requested_model', 'llama3.2-vision')}")
print(f"  Actual Model    : {naive_res.get('actual_model', naive_res.get('model', 'llama3.2-vision'))}")
print(f"  Fallback Used   : {naive_res.get('fallback_used', False)}")
if naive_res.get('fallback_used'):
    print(f"  Fallback Reason : {naive_res.get('fallback_reason')}")

print()
print("Naive Prompt Output:")
print(naive_res.get("raw_response", ""))


[VLM Runtime Notice]: llama3.2-vision encountered error (llama-server process has terminated: exit status 1: error lo...). Rerouting to llava:7b...
VLM Execution Provenance:
  Requested Model : llama3.2-vision
  Actual Model    : llava:7b
  Fallback Used   : True
  Fallback Reason : Rerouted from llama3.2-vision to llava:7b due to local engine error (llama-server process has terminated: exit status 1: error loading model: unknown model architecture: 'mllama'
error loading model: unknown model architecture: 'mllama' (status code: 500))

Naive Prompt Output:
The image appears to be a microscopy photograph showing a cluster of small, spherical particles against a dark background. These particles are likely cells or bacteria, as indicated by their size and shape. The particles are densely packed and exhibit a uniform color, which could suggest a common type or species. There is no text present in the image to provide additional context or information. The dark background enhances the visib

## 2. Structured Descriptive Prompting with llama3.2-vision
Here, llama3.2-vision is anchored as purely descriptive (prohibiting ungrounded diagnosis) and forced into a strict JSON schema permitting 'uncertain'.


In [10]:
structured_res = run_structured_vlm(sample_img, model="llama3.2-vision", temperature=0.0)
print(f"Structured VLM Execution Provenance (Actual Model: {structured_res.get('actual_model', structured_res.get('model'))} | Fallback Used: {structured_res.get('fallback_used', False)})")
print(f"Schema Validation Status:")
print(f"  json_valid      : {structured_res.get('json_valid', True)}")
print(f"  schema_complete : {structured_res.get('schema_complete', True)}")
print()
print("Structured VLM JSON Output:")
print(json.dumps(structured_res.get("json", {}), indent=2))


[VLM Runtime Notice]: llama3.2-vision encountered error (llama-server process has terminated: exit status 1: error lo...). Rerouting to llava:7b...
Structured VLM Execution Provenance (Actual Model: llava:7b | Fallback Used: True)
Schema Validation Status:
  json_valid      : True
  schema_complete : True

Structured VLM JSON Output:
{
  "modality": "fluorescence microscopy",
  "tissue_type": "cellular nuclei",
  "notable_features": [
    "signal intensity",
    "spatial distribution",
    "clustering"
  ],
  "image_quality": "high",
  "uncertainty": "The image appears to be of high quality, but the specific details of the tissue section or culture are not clearly visible.",
  "summary_narrative": "The image is a microscopic view of cellular nuclei, with a focus on signal intensity, spatial distribution, and clustering. The image quality is high, but the details of the tissue section or culture are uncertain."
}


## 3. Stochasticity vs Determinism Analysis
We evaluate repeated runs at temperature=0.7 (showing output variability) vs temperature=0.0 (showing strict determinism).


In [11]:
stoch_runs = evaluate_vlm_stochasticity(sample_img, prompt=NAIVE_PROMPT, model="llama3.2-vision", n_runs=3, temperature=0.7)
for r in stoch_runs:
    print(f"--- Run {r['run_index']} (Temp 0.7 | Model: {r.get('model')}) ---")
    print(r['response'][:200], "...\n")

det_runs = evaluate_vlm_stochasticity(sample_img, prompt=STRUCTURED_VLM_PROMPT, model="llama3.2-vision", n_runs=2, temperature=0.0)
for r in det_runs:
    print(f"--- Run {r['run_index']} (Temp 0.0 | Model: {r.get('model')}) ---")
    print(r['response'][:200], "...\n")


[VLM Runtime Notice]: llama3.2-vision encountered error (llama-server process has terminated: exit status 1: error lo...). Rerouting to llava:7b...
[VLM Runtime Notice]: llama3.2-vision encountered error (llama-server process has terminated: exit status 1: error lo...). Rerouting to llava:7b...
[VLM Runtime Notice]: llama3.2-vision encountered error (llama-server process has terminated: exit status 1: error lo...). Rerouting to llava:7b...
--- Run 1 (Temp 0.7 | Model: llama3.2-vision) ---
The image you've provided appears to be a microscopy image, likely taken with a scanning electron microscope (SEM) or a similar type of microscope. It shows a collection of spherical particles that ar ...

--- Run 2 (Temp 0.7 | Model: llama3.2-vision) ---
The image appears to be a microscopy photograph of cells, possibly bacteria, taken under fluorescence microscopy. The cells are small and round, with a blue hue, indicating they are stained with a flu ...

--- Run 3 (Temp 0.7 | Model: llama3.2-vision